# RAID v4: Semantic Precision Memory Curves

**Insight from v3:** Token-level accuracy is flat beyond ~4 tokens of context — all the context-scaling was vocabulary estimation. But long-range coherence doesn't operate at the token level. It operates at the *semantic* level — keeping you in the right "city," not predicting the specific "restaurant."

**New metrics:**
1. **Semantic similarity of top-k predictions** — how close in embedding space are the model's top predictions to the actual token? Long-range context should push predictions into the right semantic neighborhood.
2. **Expected embedding similarity** — the probability-weighted average embedding of the model's output distribution vs. the actual token's embedding.
3. **Content-word accuracy** — top-k accuracy filtered to content words only (nouns, verbs, adjectives), excluding function words that are predictable from local syntax alone.

**All three should be sensitive to "which city are you in" rather than "which restaurant did you visit."**

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import trapezoid
from pathlib import Path
import json, math, time, gc, os, torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_v4")
    if (DRIVE_DATA / "raid_corpus.jsonl").exists():
        DATA_DIR = DRIVE_DATA
        print(f"Found corpus on Drive: {DATA_DIR}")
    else:
        LOCAL_DATA = Path("/content/data/raid_sampled")
        if not (LOCAL_DATA / "raid_corpus.jsonl").exists():
            print("Corpus not found on Drive. Upload raid_corpus.jsonl:")
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
        print(f"Using local corpus: {DATA_DIR}")
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/raid_v4")
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

WINDOWS = [4, 8, 12, 16, 24, 32, 48, 64, 96, 128]
BURN_IN = 128
MAX_SCORE_TOKENS = 64
BUFFER = 10
MIN_TOKENS = BURN_IN + MAX_SCORE_TOKENS + BUFFER

TOP_K = 50  # for semantic similarity of top-k
N_RANDOM_CONTROLS = 30
RANDOM_SEED = 42

DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']
COLORS_POP = {'human': '#3498db', 'ai': '#e74c3c'}
COLORS_CTRL = {'human': '#3498db', 'ai': '#e74c3c', 'shuffled': '#2ecc71', 'uniform': '#9b59b6'}

print(f"Windows: {WINDOWS}")
print(f"Min tokens: {MIN_TOKENS}")

In [ ]:
corpus_path = DATA_DIR / "raid_corpus.jsonl"
corpus = []
with open(corpus_path) as f:
    for line in f:
        corpus.append(json.loads(line))
print(f"Loaded {len(corpus)} documents")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()

# Extract the embedding matrix for semantic similarity computation
# This is the input embedding (shared with output in most models)
embedding_matrix = model.get_input_embeddings().weight.detach()  # [vocab_size, hidden_dim]
# Normalize for cosine similarity
embedding_matrix_normed = F.normalize(embedding_matrix, dim=-1)
print(f"Embedding matrix: {embedding_matrix.shape}")

# Build a set of content-word token IDs
# Simple heuristic: tokens that start with a capital letter or are 4+ chars
# and are NOT common function words
FUNCTION_WORDS = set(tokenizer.encode(
    "the a an is are was were be been being have has had do does did will would shall should "
    "may might can could of in to for with on at by from as into through during before after "
    "above below between out off over under again further then once here there when where why "
    "how all both each few more most other some such no nor not only own same so than too very "
    "and but or if while that this these those it its he she they we you I me him her them us "
    "my your his their our who whom which what", add_special_tokens=False
))
print(f"Function word token IDs: {len(FUNCTION_WORDS)}")

print("Model loaded, embeddings extracted")

In [ ]:
@torch.no_grad()
def compute_metrics_on_region(token_ids, target_start, target_end):
    """Compute semantic and accuracy metrics on target region."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return None

    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]

    # Metrics accumulators
    total_loss = 0.0
    top10_hits = 0
    top10_content_hits = 0
    content_count = 0

    # Semantic metrics
    topk_sim_sum = 0.0       # mean cosine sim of top-k predictions to actual token
    expected_sim_sum = 0.0   # prob-weighted embedding similarity to actual token
    content_topk_sim_sum = 0.0
    count = 0

    for i in range(target_start, target_end - 1):
        target_token = token_ids[i + 1]
        token_logits = logits[i]

        # Perplexity
        log_probs = torch.log_softmax(token_logits, dim=-1)
        total_loss += -log_probs[target_token].item()

        # Top-10 accuracy
        top10_ids = torch.topk(token_logits, 10).indices
        if target_token in top10_ids:
            top10_hits += 1

        # --- Semantic similarity of top-k predictions to actual token ---
        topk_result = torch.topk(token_logits, TOP_K)
        topk_ids = topk_result.indices

        # Cosine similarity between each top-k token embedding and actual token embedding
        actual_emb = embedding_matrix_normed[target_token]  # [hidden_dim]
        topk_embs = embedding_matrix_normed[topk_ids]  # [k, hidden_dim]
        sims = torch.mv(topk_embs, actual_emb)  # [k] cosine similarities
        topk_sim_sum += sims.mean().item()

        # --- Expected embedding similarity (prob-weighted) ---
        # Use top-200 tokens for efficiency (covers >95% of probability mass)
        top200 = torch.topk(token_logits, 200)
        top200_probs = torch.softmax(top200.values, dim=-1)  # [200]
        top200_embs = embedding_matrix_normed[top200.indices]  # [200, hidden_dim]
        # Weighted average embedding
        expected_emb = torch.mv(top200_embs.t(), top200_probs)  # [hidden_dim]
        expected_emb = F.normalize(expected_emb, dim=-1)
        expected_sim_sum += torch.dot(expected_emb, actual_emb).item()

        # --- Content word metrics ---
        is_content = target_token not in FUNCTION_WORDS
        if is_content:
            content_count += 1
            if target_token in top10_ids:
                top10_content_hits += 1
            content_topk_sim_sum += sims.mean().item()

        count += 1

    del outputs, logits
    torch.cuda.empty_cache()

    if count == 0:
        return None

    result = {
        'ppl': math.exp(total_loss / count),
        'top10_acc': top10_hits / count,
        'topk_sim': topk_sim_sum / count,
        'expected_sim': expected_sim_sum / count,
        'count': count,
        'content_count': content_count,
    }
    if content_count > 0:
        result['content_top10_acc'] = top10_content_hits / content_count
        result['content_topk_sim'] = content_topk_sim_sum / content_count

    return result


def compute_memory_curve(token_ids, windows, burn_in, max_score_tokens):
    """Compute all metrics at each context window size."""
    n_tokens = len(token_ids)
    if n_tokens <= burn_in:
        return None
    target_end = min(n_tokens, burn_in + max_score_tokens)

    results = {}
    for W in windows:
        context_start = max(0, burn_in - W)
        actual_context = burn_in - context_start
        if actual_context < 4:
            continue
        truncated = token_ids[context_start:target_end]
        metrics = compute_metrics_on_region(truncated, actual_context, len(truncated))
        if metrics is not None and not math.isinf(metrics['ppl']):
            results[W] = metrics

    return results if len(results) >= 3 else None


def compute_half_life(windows, values, increasing=True):
    """Half-life from arrays of windows and values."""
    if len(windows) < 2:
        return float('nan')
    w = np.array(windows)
    v = np.array(values)
    if increasing:
        total = v[-1] - v[0]
        if total <= 0: return float('nan')
        target = v[0] + 0.5 * total
        for i in range(len(v) - 1):
            if v[i] <= target <= v[i + 1]:
                frac = (target - v[i]) / (v[i + 1] - v[i])
                return w[i] + frac * (w[i + 1] - w[i])
    else:
        total = v[0] - v[-1]
        if total <= 0: return float('nan')
        target = v[0] - 0.5 * total
        for i in range(len(v) - 1):
            if v[i] >= target >= v[i + 1]:
                frac = (v[i] - target) / (v[i] - v[i + 1])
                return w[i] + frac * (w[i + 1] - w[i])
    return w[-1]


METRICS = ['ppl', 'top10_acc', 'topk_sim', 'expected_sim', 'content_top10_acc', 'content_topk_sim']
METRIC_INCREASING = {
    'ppl': False, 'top10_acc': True, 'topk_sim': True,
    'expected_sim': True, 'content_top10_acc': True, 'content_topk_sim': True,
}


def extract_row(doc, curve_results, n_tokens):
    """Extract all metrics from curve results."""
    row = {
        'doc_id': doc['doc_id'],
        'domain': doc['domain'],
        'model': doc['model'],
        'population': doc['population'],
        'token_count': n_tokens,
    }

    sorted_W = sorted(curve_results.keys())

    for W, metrics in sorted(curve_results.items()):
        for m in METRICS:
            if m in metrics:
                row[f'{m}_W{W}'] = metrics[m]

    # Half-lives for each metric
    for m in METRICS:
        vals = [curve_results[w].get(m, np.nan) for w in sorted_W]
        if not any(np.isnan(v) for v in vals):
            row[f'hl_{m}'] = compute_half_life(sorted_W, vals, METRIC_INCREASING[m])

    return row


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "essay_results_v4.csv"

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded existing results: {len(df)} rows")
else:
    results = []
    skipped = 0
    for doc in tqdm(corpus, desc="Essays"):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        n_tokens = len(token_ids)
        if n_tokens < MIN_TOKENS:
            skipped += 1
            continue
        curve = compute_memory_curve(token_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
        if curve is None:
            skipped += 1
            continue
        row = extract_row(doc, curve, n_tokens)
        results.append(row)

    df = pd.DataFrame(results)
    df.to_csv(results_path, index=False)
    print(f"Processed {len(df)} essays ({skipped} skipped)")
    print(f"Saved to {results_path}")

print(f"Human: {len(df[df.population == 'human'])}, AI: {len(df[df.population == 'ai'])}")

In [ ]:
controls_path = BASE_DIR / "control_results_v4.csv"

if controls_path.exists():
    df_ctrl = pd.read_csv(controls_path)
    print(f"Loaded existing controls: {len(df_ctrl)} rows")
else:
    rng = np.random.RandomState(RANDOM_SEED)
    human_docs = [d for d in corpus if d['population'] == 'human']
    sample_docs = rng.choice(human_docs, size=min(N_RANDOM_CONTROLS, len(human_docs)), replace=False)

    ctrl_rows = []

    rng_s = np.random.RandomState(RANDOM_SEED)
    for i, doc in enumerate(tqdm(sample_docs, desc="Shuffled")):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < MIN_TOKENS:
            continue
        shuffled = list(token_ids)
        rng_s.shuffle(shuffled)
        curve = compute_memory_curve(shuffled, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
        if curve is None:
            continue
        ctrl_doc = {'doc_id': f'shuffled_{i:03d}', 'domain': 'shuffled', 'model': 'shuffled', 'population': 'shuffled'}
        ctrl_rows.append(extract_row(ctrl_doc, curve, len(token_ids)))

    rng_u = np.random.RandomState(RANDOM_SEED + 1)
    vocab_size = tokenizer.vocab_size
    rng_len = np.random.RandomState(RANDOM_SEED)
    sample_docs2 = rng_len.choice(human_docs, size=min(N_RANDOM_CONTROLS, len(human_docs)), replace=False)
    for i, doc in enumerate(tqdm(sample_docs2, desc="Uniform")):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < MIN_TOKENS:
            continue
        uniform_ids = rng_u.randint(0, vocab_size, size=len(token_ids)).tolist()
        curve = compute_memory_curve(uniform_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
        if curve is None:
            continue
        ctrl_doc = {'doc_id': f'uniform_{i:03d}', 'domain': 'uniform', 'model': 'uniform', 'population': 'uniform'}
        ctrl_rows.append(extract_row(ctrl_doc, curve, len(token_ids)))

    df_ctrl = pd.DataFrame(ctrl_rows)
    df_ctrl.to_csv(controls_path, index=False)
    print(f"Controls: {len(df_ctrl)} rows")
    print(f"Saved to {controls_path}")

print(f"Shuffled: {len(df_ctrl[df_ctrl.population == 'shuffled'])}, Uniform: {len(df_ctrl[df_ctrl.population == 'uniform'])}")

In [ ]:
# Main comparison figure: all metrics, all conditions
plot_metrics = [
    ('topk_sim', 'Top-k Semantic Similarity', True),
    ('expected_sim', 'Expected Embedding Similarity', True),
    ('content_top10_acc', 'Content-Word Top-10 Accuracy', True),
    ('content_topk_sim', 'Content-Word Semantic Similarity', True),
    ('top10_acc', 'Top-10 Accuracy (all tokens)', True),
    ('ppl', 'Perplexity (reference)', False),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for idx, (metric, label, increasing) in enumerate(plot_metrics):
    ax = axes[idx // 3, idx % 3]
    cols = [f'{metric}_W{w}' for w in WINDOWS]

    for pop, plabel, color, ls, lw, src_df in [
        ('human', 'Human', COLORS_CTRL['human'], '-', 2.5, df),
        ('ai', 'AI', COLORS_CTRL['ai'], '--', 1.5, df),
        ('shuffled', 'Shuffled', COLORS_CTRL['shuffled'], ':', 2, df_ctrl),
        ('uniform', 'Uniform', COLORS_CTRL['uniform'], ':', 2, df_ctrl),
    ]:
        sub = src_df[src_df.population == pop]
        avail = [c for c in cols if c in sub.columns]
        if len(avail) == 0 or len(sub) == 0:
            continue
        means = np.array([sub[c].mean() for c in avail])
        marker = 'o' if pop in ['human', 'ai'] else 's'
        ax.plot(WINDOWS[:len(means)], means, f'{marker}{ls}', color=color,
                linewidth=lw, markersize=4, label=plabel)

    ax.set_xscale('log', base=2)
    ax.set_xlabel('Context Window')
    ax.set_ylabel(label)
    ax.set_title(label, fontweight='bold', fontsize=11)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

plt.suptitle('Raw Metric Curves: All Conditions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_raw_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Normalized curves for semantic metrics (the key question)
sem_metrics = [
    ('topk_sim', 'Top-k Semantic Sim', True),
    ('expected_sim', 'Expected Embedding Sim', True),
    ('content_topk_sim', 'Content-Word Semantic Sim', True),
    ('content_top10_acc', 'Content-Word Top-10 Acc', True),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for idx, (metric, label, increasing) in enumerate(sem_metrics):
    ax = axes[idx]
    cols = [f'{metric}_W{w}' for w in WINDOWS]

    for pop, plabel, color, ls, lw, src_df in [
        ('human', 'Human', COLORS_CTRL['human'], '-', 2.5, df),
        ('ai', 'AI', COLORS_CTRL['ai'], '--', 1.5, df),
        ('shuffled', 'Shuffled', COLORS_CTRL['shuffled'], ':', 2, df_ctrl),
        ('uniform', 'Uniform', COLORS_CTRL['uniform'], ':', 2, df_ctrl),
    ]:
        sub = src_df[src_df.population == pop]
        avail = [c for c in cols if c in sub.columns]
        if len(avail) == 0 or len(sub) == 0:
            continue
        means = np.array([sub[c].mean() for c in avail])
        if increasing:
            total = means[-1] - means[0]
            if total > 0.001:
                norm = (means - means[0]) / total
            else:
                continue
        else:
            total = means[0] - means[-1]
            if total > 0.001:
                norm = (means[0] - means) / total
            else:
                continue

        marker = 'o' if pop in ['human', 'ai'] else 's'
        ax.plot(WINDOWS[:len(norm)], norm, f'{marker}{ls}', color=color,
                linewidth=lw, markersize=4, label=plabel)

    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.plot([WINDOWS[0], WINDOWS[-1]], [0, 1], 'k--', alpha=0.2, label='Linear')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('Context Window')
    ax.set_ylabel('Fraction of Total Benefit')
    ax.set_title(label, fontweight='bold', fontsize=11)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

plt.suptitle('Normalized Semantic Curves: Does Long-Range Context Help Semantic Precision?',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig2_normalized_semantic.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Semantic similarity by genre (human only) — the universality test
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, domain in enumerate(DOMAINS):
    ax = axes[idx // 4, idx % 4]
    cols = [f'topk_sim_W{w}' for w in WINDOWS]

    for pop, ls, lw in [('human', '-', 2.5), ('ai', '--', 1.5)]:
        sub = df[(df.domain == domain) & (df.population == pop)]
        if len(sub) == 0:
            continue
        avail = [c for c in cols if c in sub.columns]
        means = np.array([sub[c].mean() for c in avail])
        total = means[-1] - means[0]
        if total > 0.001:
            norm = (means - means[0]) / total
            ax.plot(WINDOWS[:len(norm)], norm, marker='o', linestyle=ls, linewidth=lw,
                    color=COLORS_POP[pop], markersize=4, label=pop)

    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(domain, fontsize=12, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.05, 1.05)
    if idx % 4 == 0:
        ax.set_ylabel('Fraction of Total Sim Gain')
    if idx >= 4:
        ax.set_xlabel('Context Window')
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8)

plt.suptitle('Normalized Semantic Similarity by Genre: Human vs AI',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig3_semantic_by_genre.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
print("="*70)
print("SUMMARY: Semantic Precision Memory Curves")
print("="*70)

for metric, label, increasing in [
    ('topk_sim', 'Top-k Semantic Sim', True),
    ('expected_sim', 'Expected Embedding Sim', True),
    ('content_topk_sim', 'Content Semantic Sim', True),
    ('content_top10_acc', 'Content Top-10 Acc', True),
    ('top10_acc', 'Top-10 Acc (all)', True),
    ('ppl', 'Perplexity', False),
]:
    print(f"\n--- {label} ---")
    print(f"  {'Condition':<12} {'W4':>8} {'W128':>8} {'Delta':>8} {'Half-life':>10}")
    print(f"  {'-'*50}")
    for pop, plabel, src in [('human','Human',df), ('ai','AI',df),
                              ('shuffled','Shuffled',df_ctrl), ('uniform','Uniform',df_ctrl)]:
        sub = src[src.population == pop]
        c4 = f'{metric}_W4'
        c128 = f'{metric}_W128'
        hl_col = f'hl_{metric}'
        if c4 not in sub.columns or len(sub) == 0:
            continue
        w4 = sub[c4].mean()
        w128 = sub[c128].mean() if c128 in sub.columns else np.nan
        delta = w128 - w4 if increasing else w4 - w128
        hl = sub[hl_col].dropna().mean() if hl_col in sub.columns else np.nan
        print(f"  {plabel:<12} {w4:>8.4f} {w128:>8.4f} {delta:>8.4f} {hl:>10.1f}")

print("\n\n--- Genre half-lives (human only, semantic sim) ---")
h = df[df.population == 'human']
for metric, label in [('topk_sim', 'Top-k Sem Sim'), ('content_topk_sim', 'Content Sem Sim')]:
    print(f"\n  {label}:")
    hl_col = f'hl_{metric}'
    if hl_col not in h.columns:
        continue
    for d in DOMAINS:
        sub = h[h.domain == d]
        hl = sub[hl_col].dropna()
        print(f"    {d:<15} {hl.mean():>8.1f} +/- {hl.std():>6.1f}")

    groups = [h[h.domain == d][hl_col].dropna() for d in DOMAINS if len(h[h.domain == d][hl_col].dropna()) > 2]
    if len(groups) >= 2:
        f_val, p_val = stats.f_oneway(*groups)
        print(f"    ANOVA: F={f_val:.3f}, p={p_val:.4f}")

    hh = df[df.population == 'human'][hl_col].dropna()
    aa = df[df.population == 'ai'][hl_col].dropna()
    if len(hh) > 0 and len(aa) > 0:
        t, p = stats.ttest_ind(hh, aa)
        d_val = (hh.mean() - aa.mean()) / np.sqrt((hh.std()**2 + aa.std()**2) / 2)
        print(f"    Human vs AI: d={d_val:.3f}, p={p:.4f}")